# 02. 전처리

- **목적**: `01` 6절에서 찾은 문제 처리, 커피-음료만 남겨 SQL 적재용 테이블 3개 저장
- **데이터 출처** (`data/raw/`, cp949)
  - `서울시_상권분석서비스(추정매출-상권)_*.csv` 5개, `서울시_상권분석서비스(점포-상권)_*.csv` 5개, `서울시_상권분석서비스(영역-상권).csv`
  - 기간: 2021년 1분기 ~ 2026년 2분기 (22개 분기)
- **출력**: `data/processed/커피_매출.csv`, `커피_점포.csv`, `영역.csv`
- **작성일**: 2026-10-16 (출력은 2026-10-15 실행 결과)

## 결론 요약
- 커피 매출 23,700행 × 55열, 커피 점포 33,263행 × 14열, 영역 1,650행 × 11열 저장
- 키 중복, 상권 이름 결측, `?` 남은 이름, 이름 2개인 상권, 깨진 시간대 컬럼, float 컬럼 모두 0건 확인
- 세 테이블은 합치지 않고 따로 저장, 연결은 SQL에서 처리

## 설정

In [1]:
import sys
from pathlib import Path

import pandas as pd

sys.path.append("..")  # notebooks/에서 실행하므로 프로젝트 루트의 src를 import 경로에 추가
from src.data_io import CODE_DTYPES, load_yearly_csv

pd.set_option("display.max_columns", 20)
pd.set_option("display.max_rows", 60)

In [2]:
RAW_DIR = Path("../data/raw")
OUT_DIR = Path("../data/processed")
AREA_FILE = "서울시_상권분석서비스(영역-상권).csv"

CAFE_CODE = "CS100010"  # 커피-음료
KEY_COLS = ["기준_년분기_코드", "상권_코드"]  # 커피 한 업종만 남기므로 업종 코드는 키에서 제외

STORE_RENAME = {"점포_수": "일반_점포_수", "유사_업종_점포_수": "전체_점포_수"}  # 2025년 파일부터 새 이름 사용
TIME_RANGES = ["00~06", "06~11", "11~14", "14~17", "17~21", "21~24"]
# 원자료 건수 컬럼명이 '시간대_건수~06_매출_건수'처럼 시작 시각이 빠져 있어 금액 컬럼 규칙에 맞춤
SALES_RENAME = {f"시간대_건수~{t[-2:]}_매출_건수": f"시간대_{t}_매출_건수" for t in TIME_RANGES}

## 데이터 로드와 검증

- 연도별 파일을 합치면서 컬럼명을 통일 (점포: 2025년 변경분, 매출: 깨진 시간대 건수 컬럼)

In [3]:
raw_store = load_yearly_csv(RAW_DIR, "*점포-상권*.csv", rename_cols=STORE_RENAME)

서울시_상권분석서비스(점포-상권)_2021년.csv (303880, 14)


서울시_상권분석서비스(점포-상권)_2022년.csv (305587, 14)


서울시_상권분석서비스(점포-상권)_2023년.csv (307741, 14)


서울시_상권분석서비스(점포-상권)_2024년.csv (306889, 14)


서울시_상권분석서비스(점포-상권)_2025~2026년.csv (456659, 14)


In [4]:
raw_sales = load_yearly_csv(RAW_DIR, "*추정매출*.csv", rename_cols=SALES_RENAME)

서울시_상권분석서비스(추정매출-상권)_2021년.csv (89150, 55)


서울시_상권분석서비스(추정매출-상권)_2022년.csv (88834, 55)


서울시_상권분석서비스(추정매출-상권)_2023년.csv (88246, 55)


서울시_상권분석서비스(추정매출-상권)_2024년.csv (87179, 55)


서울시_상권분석서비스(추정매출-상권)_2025~2026년.csv (128053, 55)


In [5]:
raw_area = pd.read_csv(RAW_DIR / AREA_FILE, encoding="cp949", dtype=CODE_DTYPES)

In [6]:
# 점포 컬럼명 통일이 맞다면 모든 행에서 일반 + 프랜차이즈 = 전체 (01 1절에서 확인한 조건)
assert (raw_store["일반_점포_수"] + raw_store["프랜차이즈_점포_수"] == raw_store["전체_점포_수"]).all()
# 이름 통일을 영역 테이블 기준으로 하므로 모든 상권 코드가 영역에 있어야 함
assert set(raw_sales["상권_코드"]) <= set(raw_area["상권_코드"])
assert set(raw_store["상권_코드"]) <= set(raw_area["상권_코드"])
assert not any(col.startswith("시간대_건수") for col in raw_sales.columns)

## 전처리

- 영역: `?`로 깨진 가운뎃점 복구
- 매출·점포: 커피만 남기고, 상권 이름을 영역 테이블 기준으로 통일, 키 순서로 정렬
- 매출: 소수 부분이 없는 float 금액 컬럼을 int로 변환 (SQL `BIGINT` 적재용)

In [7]:
def clean_area(raw_area: pd.DataFrame) -> pd.DataFrame:
    """Restore the middle dot (·) that was broken into '?' in area and dong names."""
    return raw_area.assign(
        상권_코드_명=lambda d: d["상권_코드_명"].str.replace("?", "·", regex=False),
        행정동_코드_명=lambda d: d["행정동_코드_명"].str.replace("?", "·", regex=False),
    )


def prepare_cafe_table(raw: pd.DataFrame, industry_code: str, area_names: pd.Series,
                       sort_keys: list[str]) -> pd.DataFrame:
    """Keep one industry, take area names from the area table, and sort by key."""
    return (
        raw.loc[lambda d: d["서비스_업종_코드"] == industry_code]
        .assign(상권_코드_명=lambda d: d["상권_코드"].map(area_names))
        .sort_values(sort_keys)
        .reset_index(drop=True)
    )


def cast_whole_floats_to_int(df: pd.DataFrame) -> pd.DataFrame:
    """Cast float columns to int64, failing if any value has a fractional part."""
    float_cols = df.select_dtypes("float").columns
    assert (df[float_cols] % 1 == 0).all().all(), "fractional values found"
    return df.astype({col: "int64" for col in float_cols})

In [8]:
area = clean_area(raw_area)
area_names = area.set_index("상권_코드")["상권_코드_명"]

cafe_store = prepare_cafe_table(raw_store, CAFE_CODE, area_names, KEY_COLS)
cafe_sales = prepare_cafe_table(raw_sales, CAFE_CODE, area_names, KEY_COLS).pipe(cast_whole_floats_to_int)
print("커피 매출", len(cafe_sales), "/ 커피 점포", len(cafe_store))

커피 매출 23700 / 커피 점포 33263


## 검증

- `01`에서 찾은 문제가 모두 0건인지 확인, 하나라도 남으면 assert에서 중단

In [9]:
n_broken_names = (area["상권_코드_명"].str.contains("?", regex=False).sum()
                  + area["행정동_코드_명"].str.contains("?", regex=False).sum())

checks = {
    "매출 키 중복": cafe_sales.duplicated(KEY_COLS).sum(),
    "점포 키 중복": cafe_store.duplicated(KEY_COLS).sum(),
    "상권 이름 빈 값": cafe_sales["상권_코드_명"].isna().sum() + cafe_store["상권_코드_명"].isna().sum(),
    "이름에 '?' 남음": n_broken_names,
    "이름이 2개인 상권": (cafe_sales.groupby("상권_코드")["상권_코드_명"].nunique() > 1).sum(),
    "깨진 시간대 열": sum(col.startswith("시간대_건수") for col in cafe_sales.columns),
    "실수형 열": len(cafe_sales.select_dtypes("float").columns),
}
assert all(v == 0 for v in checks.values()), checks
checks

{'매출 키 중복': np.int64(0),
 '점포 키 중복': np.int64(0),
 '상권 이름 빈 값': np.int64(0),
 "이름에 '?' 남음": np.int64(0),
 '이름이 2개인 상권': np.int64(0),
 '깨진 시간대 열': 0,
 '실수형 열': 0}

### 관찰
- 7개 점검 항목 모두 0건 → `01` 6절의 문제 모두 처리 확인

## 저장

- 세 테이블을 따로 저장, 연결은 `03` 이후 SQL에서 처리
- 저장 후 다시 읽어 행 수가 같은지 확인 (인코딩·구분자 문제로 행이 깨지는 경우 대비)

In [10]:
OUT_DIR.mkdir(parents=True, exist_ok=True)

outputs = {"커피_매출": cafe_sales, "커피_점포": cafe_store, "영역": area}
for name, table in outputs.items():
    path = OUT_DIR / f"{name}.csv"
    table.to_csv(path, index=False, encoding="utf-8")
    reloaded = pd.read_csv(path, dtype=CODE_DTYPES)
    print(name, table.shape, "| 다시 읽은 행 수 일치:", len(reloaded) == len(table))
    assert len(reloaded) == len(table)

커피_매출 (23700, 55) | 다시 읽은 행 수 일치: True
커피_점포 (33263, 14) | 다시 읽은 행 수 일치: True
영역 (1650, 11) | 다시 읽은 행 수 일치: True


## 결론과 다음 단계

- 커피 매출 23,700행, 커피 점포 33,263행, 영역 1,650행 저장, 다시 읽은 행 수 모두 일치
- `01`에서 찾은 문제 7개 항목 모두 0건
- 다음 단계: `03_SQL적재.ipynb`에서 PostgreSQL 적재